# 🛒 고객 문의 유형 분류기
**유형:** 분류형 (임베딩 + 로지스틱 회귀)  
**목표:** 고객 문의 내용을 입력하면 어떤 유형(AS, 결제, 교환, 반품, 배송, 업무처리, 주문)인지 예측하는 AI 만들기

---
### 데이터 출처
- **K쇼핑 콜센터 질의응답 데이터** (AI Hub 민원 콜센터 데이터셋)
- 7개 카테고리: AS, 결제, 교환, 반품, 배송, 업무처리, 주문
- 고객 질문만 추출하여 분류기 학습

### 이 노트북의 구조
| 단계 | 내용 |
|------|------|
| Step 1 | 데이터 로드 & 전처리 |
| Step 2 | 임베딩 & 분류기 학습 |
| Step 3 | 예측 & 평가 |
| Step 4 | 미니 미션 |

## Step 1: 데이터 로드 & 전처리

In [ ]:
# 1. 필수 라이브러리 설치
!pip install sentence-transformers scikit-learn torch

In [ ]:
# 2. K쇼핑 콜센터 데이터 로드 (zip → JSON → DataFrame)
import json
import zipfile
import os
import pandas as pd
from pathlib import Path

# 데이터 경로
train_dir = Path("../Data/1.Training/라벨링데이터_231222_add")
val_dir = Path("../Data/2.Validation/라벨링데이터_231222_add")

# K쇼핑 카테고리 목록
categories = ["AS", "결제", "교환", "반품", "배송", "업무처리", "주문"]

def load_kshopping_data(base_dir, split_name):
    """K쇼핑 zip 파일에서 고객 질문만 추출"""
    all_questions = []
    
    for cat in categories:
        zip_name = f"민원(콜센터) 질의응답_K쇼핑_{cat}_{split_name}.zip"
        zip_path = base_dir / zip_name
        
        if not zip_path.exists():
            print(f"⚠️ 파일 없음: {zip_name}")
            continue
        
        with zipfile.ZipFile(zip_path, 'r') as zf:
            for fname in zf.namelist():
                with zf.open(fname) as f:
                    data = json.load(f)
        
        # 고객 질문만 필터링 (QA=="Q" & 고객질문 존재)
        questions = [
            {"text": d["고객질문(요청)"].strip(), "category": cat}
            for d in data
            if d.get("QA") == "Q" and d.get("고객질문(요청)", "").strip()
        ]
        all_questions.extend(questions)
        print(f"  ✅ {cat}: {len(questions)}건")
    
    return pd.DataFrame(all_questions)

print("🔹 Training 데이터 로드 중...")
train_full_df = load_kshopping_data(train_dir, "Training")
print(f"\n📊 Training 전체: {len(train_full_df)}건")

print("\n🔹 Validation 데이터 로드 중...")
val_full_df = load_kshopping_data(val_dir, "Validation")
print(f"\n📊 Validation 전체: {len(val_full_df)}건")

In [ ]:
# 3. 카테고리별 분포 확인
print("=== Training 카테고리 분포 ===")
print(train_full_df["category"].value_counts())
print(f"\n총 카테고리 수: {train_full_df['category'].nunique()}")

In [ ]:
# 4. 실습용 샘플링 (시간 절약)
# 카테고리별 균등 샘플링: 각 카테고리에서 최대 500건씩
SAMPLE_PER_CAT = 500

train_df = train_full_df.groupby("category").apply(
    lambda x: x.sample(n=min(len(x), SAMPLE_PER_CAT), random_state=42)
).reset_index(drop=True)

test_df = val_full_df.groupby("category").apply(
    lambda x: x.sample(n=min(len(x), SAMPLE_PER_CAT // 5), random_state=42)
).reset_index(drop=True)

print(f"샘플링 후 train: {len(train_df)}건")
print(f"샘플링 후 test: {len(test_df)}건")
print(f"\n=== Train 카테고리별 분포 ===")
print(train_df["category"].value_counts().sort_index())
print(f"\n=== Test 카테고리별 분포 ===")
print(test_df["category"].value_counts().sort_index())

In [ ]:
# 5. 샘플 데이터 확인
print("=== 카테고리별 질문 예시 ===")
for cat in categories:
    samples = train_df[train_df["category"] == cat]["text"].head(2).tolist()
    print(f"\n🏷️ [{cat}]")
    for s in samples:
        print(f"   → {s}")

---
## Step 2: 임베딩 & 분류기 학습

In [ ]:
# 6. 라벨 인코딩
from sklearn.preprocessing import LabelEncoder

label_encoder = LabelEncoder()
train_df["label"] = label_encoder.fit_transform(train_df["category"])
test_df["label"]  = label_encoder.transform(test_df["category"])

print("카테고리 → 숫자 매핑:")
for i, cls in enumerate(label_encoder.classes_):
    print(f"  {i}: {cls}")

In [ ]:
# 7. 문장 임베딩
# ⏳ 이 셀은 3~5분 걸립니다!
from sentence_transformers import SentenceTransformer
import numpy as np

model = SentenceTransformer("sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2")

def embed_texts(texts, batch_size=64):
    return model.encode(texts, batch_size=batch_size, show_progress_bar=True, convert_to_numpy=True)

print("🔹 train 임베딩 중...")
X_train = embed_texts(train_df["text"].tolist())
y_train = train_df["label"].values

print("🔹 test 임베딩 중...")
X_test = embed_texts(test_df["text"].tolist())
y_test = test_df["label"].values

print(f"\n임베딩 shape: train {X_train.shape}, test {X_test.shape}")

In [ ]:
# 8. 로지스틱 회귀 분류기 학습
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report

clf = LogisticRegression(max_iter=2000)

print("🔹 고객 문의 분류기 학습 중...")
clf.fit(X_train, y_train)

y_pred = clf.predict(X_test)
acc = accuracy_score(y_test, y_pred)
print(f"\n✅ Test Accuracy: {acc:.4f}")

print("\n=== 분류 리포트 ===")
print(classification_report(y_test, y_pred, target_names=label_encoder.classes_))

---
## Step 3: 예측 & 평가

In [ ]:
# 9. 예측 함수
def predict_inquiry(text: str, top_k: int = 3):
    """고객 문의 텍스트를 입력받아 유형을 예측"""
    emb = model.encode([text], convert_to_numpy=True)
    probs = clf.predict_proba(emb)[0]
    top_indices = np.argsort(-probs)[:top_k]

    print("\n" + "=" * 50)
    print("📞 고객 문의 내용:")
    print(f"   {text}")
    print("=" * 50)
    print("\n🔮 예측된 문의 유형 (Top-k):")
    for idx in top_indices:
        cat_name = label_encoder.classes_[idx]
        p = probs[idx]
        bar = "█" * int(p * 20)
        print(f"   {cat_name:8s} {bar} ({p:.3f})")
    print("=" * 50 + "\n")

In [ ]:
# 10. 테스트: 명확한 문의
print(">>> 테스트 1: 배송 문의")
predict_inquiry("주문한 물건이 아직 안 왔어요. 배송 조회 좀 해주세요.")

print(">>> 테스트 2: 반품 문의")
predict_inquiry("이 제품 환불하고 싶어요. 반품 접수해주세요.")

print(">>> 테스트 3: 결제 문의")
predict_inquiry("카드 결제가 안 돼요. 다른 결제 수단으로 바꿀 수 있나요?")

---
## Step 4: 미니 미션 🎯

### 🎯 미션 1: 직접 고객 문의 작성하기
각 카테고리에 맞는 고객 문의를 직접 작성하고, AI가 올바르게 분류하는지 확인하세요.

In [ ]:
# 미션 1-1: AS 관련 문의
my_inquiry_1 = "TV 화면이 깨져서 나와요. AS 접수하려면 어떻게 해야 하나요?"
predict_inquiry(my_inquiry_1)
# 👉 예상한 카테고리: AS
# 👉 AI의 예측이 맞았나요?: 

In [ ]:
# 미션 1-2: 교환 관련 문의
my_inquiry_2 = "사이즈가 안 맞아서 교환하고 싶어요. 다른 사이즈로 바꿀 수 있나요?"
predict_inquiry(my_inquiry_2)
# 👉 예상한 카테고리: 교환
# 👉 AI의 예측이 맞았나요?: 

In [ ]:
# 미션 1-3: 주문 관련 문의
my_inquiry_3 = "주문을 취소하고 싶은데요. 아직 출발 전이면 취소되나요?"
predict_inquiry(my_inquiry_3)
# 👉 예상한 카테고리: 주문
# 👉 AI의 예측이 맞았나요?: 

### 🎯 미션 2: 경계가 애매한 문의 만들기
하나의 문의가 **두 가지 카테고리에 동시에 해당**할 수 있는 문장을 만들어보세요.  
예: "주문한 물건이 불량이라 교환하고 싶어요" → 주문? 교환? AS?

In [ ]:
# 미션 2-1: 교환 + 반품이 애매한 문의
ambiguous_1 = "상품이 마음에 안 들어서 다른 걸로 바꾸거나 그냥 돌려보내고 싶어요."
predict_inquiry(ambiguous_1)
# 👉 해당할 수 있는 카테고리 2개: 
# 👉 AI 예측이 적절한가요?: 

In [ ]:
# 미션 2-2: 결제 + 주문이 애매한 문의
ambiguous_2 = "주문하면서 결제했는데 금액이 이상해요. 확인 좀 해주세요."
predict_inquiry(ambiguous_2)
# 👉 해당할 수 있는 카테고리 2개: 
# 👉 AI 예측이 적절한가요?: 

In [ ]:
# 미션 2-3: 배송 + 반품이 애매한 문의
ambiguous_3 = "배송 온 물건이 파손되어 있어서 반품하려고요. 택배 다시 보내야 하나요?"
predict_inquiry(ambiguous_3)
# 👉 해당할 수 있는 카테고리 2개: 
# 👉 AI 예측이 적절한가요?: 

### 🎯 미션 3: AI 확신도 분석
`predict_inquiry`는 상위 3개 카테고리와 확률을 보여줍니다.

- 1위 확률이 0.8 이상 → AI가 **확신**하는 것
- 1위와 2위 확률이 비슷 → AI가 **헷갈리는** 것

이 두 가지 경우를 각각 찾아보세요.

In [ ]:
# 미션 3-1: AI가 확신하는 문의 (1위 확률 0.8 이상 목표)
certain_inquiry = "배송이 언제 되나요? 송장번호 알려주세요."
predict_inquiry(certain_inquiry)
# 👉 1위 확률: 

In [ ]:
# 미션 3-2: AI가 헷갈리는 문의 (1위와 2위 확률 차이 0.1 이하 목표)
uncertain_inquiry = "이 물건 문제가 있는데 바꿔주실 수 있나요? 아니면 그냥 돈 돌려받을게요."
predict_inquiry(uncertain_inquiry)
# 👉 1위와 2위의 확률 차이: 

### 🎯 미션 4: 실제 업무 활용 아이디어
이 분류기를 실제 쇼핑몰 고객센터에 적용한다면 어떻게 활용할 수 있을까요?

In [ ]:
# ✏️ 활용 아이디어를 적어보세요 (주석으로)
#
# 1. 이 분류기를 실제 고객센터에 적용하면 어떤 점이 좋을까요?
#    → 
#
# 2. 분류 정확도를 높이려면 어떤 방법이 있을까요? (2가지 이상)
#    → 
#
# 3. 7개 카테고리 외에 추가하면 좋을 카테고리가 있다면?
#    → 
#
# 4. 이 분류기의 한계점은 무엇일까요?
#    → 